# Reflection and Blogpost Writing

## Setup

In [ ]:
llm_config = {"model": "gemini-3.1-flash-lite","api_key":"","api_type":"google"}

## The task!

In [46]:
task = '''
        Write a concise but engaging blogpost about Apple products price increase. Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [47]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [48]:
reply = writer.generate_reply(messages=[{"content": task ,"role": "user"}])

In [49]:
print(reply)

{'content': '**The Premium Price Tag: Why Apple’s Costs Are Climbing**\n\nIf you’ve noticed your tech budget stretching further to afford the latest iPhone or MacBook, you aren’t alone. Apple is steadily nudging prices upward, driven by sophisticated hardware upgrades, soaring production costs, and global inflation. \n\nWhile the "Apple Tax" is a familiar pain point, the company is betting that consumers value ecosystem integration and premium build quality over bargain hunting. For loyalists, it’s a necessary trade-off for cutting-edge performance. But as the gap between entry-level and flagship models widens, Apple is forcing users to ask: is the prestige still worth the premium?', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [50]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [51]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about Apple products price increase. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**The Apple Premium: Why Your Next Upgrade Will Cost More**

Apple’s pricing strategy is shifting. As hardware becomes more sophisticated and production costs climb, the "Apple Tax" is hitting harder than ever. From record-high iPhone price tags to expensive M-series upgrades, the entry barrier is rising. 

While some argue these prices reflect top-tier engineering and ecosystem integration, others feel the sting of diminishing returns. Are we paying for true innovation, or has Apple simply mastered the art of charging more for what we’ve come to expect? As the price gap widens, users must decide: is the prestige still worth the premium?

--------------------------------------------------------------------------------
Criti

## Nested chat

In [52]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [53]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [54]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [55]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [56]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [57]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

In [58]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about Apple products price increase. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**The Price of Perfection: Why Apple is Getting Pricier**

Apple’s latest price hikes have left fans wondering: are we paying for innovation or just the brand? With global supply chain shifts and high-end manufacturing costs, Apple is betting that its ecosystem is too essential to leave. While the sticker shock is real, the company continues to focus on premium materials and powerful silicon that set industry benchmarks. 

Is the higher price tag justified by performance, or has the "Apple Tax" finally hit its limit? As competition narrows the gap, every dollar spent now comes with higher expectations. Is your next upgrade still worth it?

--------------------------------------------------------------------------------

***

## Get the summary

In [59]:
print(res.summary)

**The Evolution of Apple’s Premium Pricing Strategy**

*Disclosure: This post contains no affiliate links. All analysis is based on publicly available market reports.*

Apple’s recent pricing shifts have prompted significant discussion regarding the company’s long-term market strategy. Rather than simple inflation, these adjustments reflect a complex balance of manufacturing costs and ecosystem investment.

## Analyzing the Trend: 2023 vs. 2024
Market data from [IDC](https://www.idc.com) indicates a rise in Average Selling Price (ASP) across the flagship range. 

| Model | 2023 Base Price | 2024 Base Price |
| :--- | :--- | :--- |
| iPhone Pro Series | $999 | $999-$1,199 |
| MacBook Pro (M-Series) | Base Entry | Increased Tiering |

## Supply Chain and Production Factors
Industry analysis suggests that the transition to 3nm chip architecture and titanium chassis construction has increased bill-of-materials (BOM) costs. According to [10-K filings](https://investor.apple.com), Apple main